In [2]:
# 1. Check GPU
import torch
print("="*60)
print("GPU CHECK")
print("="*60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ GPU: {gpu_name}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"   Total GPU Memory: {total_mem:.1f} GB")
    if total_mem < 20:
        print("⚠️  Warning: Running 3 servers might be tight on memory. L4 or A100 recommended.")
else:
    print("❌ No GPU found! Please enable GPU in Runtime > Change runtime type.")
    raise RuntimeError("GPU required")

GPU CHECK
✅ GPU: NVIDIA L4
   Total GPU Memory: 22.2 GB


In [3]:
# 2. Setup Working Directories
import os

print("📦 Setting up working directories...")

# Clear PYTHONPATH to avoid interference
if 'PYTHONPATH' in os.environ:
    del os.environ['PYTHONPATH']

# Create directories
os.makedirs("/content/Evo-1/checkpoints/libero", exist_ok=True)
os.makedirs("/content/Evo-1/checkpoints/metaworld", exist_ok=True)
os.makedirs("/content/logs", exist_ok=True)

print("✅ Directories created")
print("   - /content/Evo-1/checkpoints/libero")
print("   - /content/Evo-1/checkpoints/metaworld")
print("   - /content/logs")

📦 Setting up working directories...
✅ Directories created
   - /content/Evo-1/checkpoints/libero
   - /content/Evo-1/checkpoints/metaworld
   - /content/logs


In [4]:
# 3. Install System Dependencies
print("📦 Installing system dependencies...")
!apt-get update -qq
!apt-get install -y git wget net-tools libosmesa6-dev patchelf ffmpeg libgl1-mesa-glx > /dev/null 2>&1
print("✅ System dependencies installed")

📦 Installing system dependencies...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ System dependencies installed
✅ System dependencies installed


In [ ]:
# 4. Install Dependencies (Part 1 - Core packages)
# flash-attn will be installed in a separate cell for better control

print("📦 Installing Python packages...")
print("   Using PyTorch 2.5.1+cu121\n")

# Step 1: Install PyTorch 2.5.1 + CUDA 12.1
print("📦 Step 1: Installing PyTorch 2.5.1+cu121...")
!pip install -q torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121

# Step 2: Install Evo-1 server dependencies
print("\n📦 Step 2: Installing Evo-1 server dependencies...")
!pip install -q "transformers>=4.45.0" "timm>=1.0.0" "accelerate>=1.0.0" "diffusers>=0.31.0"
!pip install -q websockets pyyaml opencv-python pillow "numpy>=1.26.4,<2.0" fvcore
!pip install -q einops thop cloudpickle easydict hydra-core
!pip install -q huggingface_hub packaging ninja

# Step 3: Install LIBERO dependencies (for client)
# CRITICAL: LIBERO requires robosuite==1.4.1 (NOT 1.5+)
# robosuite 1.5 removed single_arm_env which LIBERO depends on
print("\n📦 Step 3: Installing LIBERO client dependencies...")
!pip install -q mujoco imageio h5py
!pip install -q robosuite==1.4.1  # LIBERO requires 1.4.1, NOT newer versions
!pip install bddl

# Step 4: Install MetaWorld dependencies
print("\n📦 Step 4: Installing MetaWorld dependencies...")
!pip install -q metaworld gymnasium

# Step 5: Install VLABench dependencies
print("\n📦 Step 5: Installing VLABench dependencies...")
# Use --no-cache-dir to avoid hash mismatch issues from cached packages
!pip install -q --no-cache-dir mediapy websocket-client colorlog colorama
# Install open3d separately - sometimes has hash issues on Colab
!pip install -q --no-cache-dir open3d || pip install -q open3d --upgrade

# Step 5b: Install LeRobot and conversion dependencies
# Required for convert_to_lerobot.py script
print("\n📦 Step 5b: Installing LeRobot and conversion dependencies...")
!pip install -q scipy tqdm
# datasets==3.2.0 is required for LeRobot compatibility (per VLABench docs)
!pip install -q "datasets==3.2.0"
# Install LeRobot WITHOUT its dependencies to avoid numpy/torch conflicts
# LeRobot would otherwise upgrade numpy to 2.x and torch to 2.7.x
!pip install -q lerobot --no-deps

# Step 5c: Re-pin numpy and torch to correct versions (lerobot may have broken them)
print("\n📦 Step 5c: Re-pinning numpy and torch versions...")
!pip install -q "numpy>=1.26.4,<2.0" --force-reinstall
!pip install -q torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121 --force-reinstall

# Step 6: Pre-configure robosuite macros to avoid interactive prompt
print("\n📦 Step 6: Pre-configuring robosuite...")
import os
import yaml

robosuite_macros_dir = os.path.expanduser("~/.robosuite")
os.makedirs(robosuite_macros_dir, exist_ok=True)
macros_file = os.path.join(robosuite_macros_dir, "macros_private.py")
if not os.path.exists(macros_file):
    with open(macros_file, 'w') as f:
        f.write('# Robosuite macros - auto-generated\n')
        f.write('import robosuite.macros as macros\n')
        f.write('macros.SIMULATION_TIMESTEP = 0.002\n')
        f.write('macros.SHOW_WARNINGS = False\n')
    print("   ✅ Created robosuite macros file")
else:
    print("   ✅ Robosuite macros already configured")

# Step 7: Pre-configure LIBERO to avoid interactive prompt
print("\n📦 Step 7: Pre-configuring LIBERO...")
libero_config_dir = os.path.expanduser("~/.libero")
os.makedirs(libero_config_dir, exist_ok=True)
libero_config_file = os.path.join(libero_config_dir, "config.yaml")
if not os.path.exists(libero_config_file):
    # Create default config pointing to where LIBERO will be cloned
    libero_paths = {
        "benchmark_root": "/content/LIBERO/libero/libero",
        "bddl_files": "/content/LIBERO/libero/libero/bddl_files",
        "init_states": "/content/LIBERO/libero/libero/init_files",
        "datasets": "/content/LIBERO/datasets",
        "assets": "/content/LIBERO/libero/libero/assets",
    }
    with open(libero_config_file, 'w') as f:
        yaml.dump(libero_paths, f)
    print("   ✅ Created LIBERO config file")
else:
    print("   ✅ LIBERO config already configured")

print("\n" + "="*60)
print("✅ Core dependencies installed!")
print("   Note: robosuite==1.4.1 installed (LIBERO requirement)")
print("="*60)
print("\n➡️  Now run Cell 5 to install flash-attn (separate cell for better control)")

📦 Installing Python packages...
   Using PyTorch 2.5.1+cu121

📦 Step 1: Installing PyTorch 2.5.1+cu121...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.9 MB/s eta 0:00:0000:0100:01     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/780.4 MB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.9 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 82.4 MB/s eta 0:00:00:00:010:01     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/7.3 MB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 82.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 118.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 118.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 108.3 MB/s eta 0:00:0000:0100:01     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/23.7 MB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 108.3 MB/s eta 0:00:0000:01

In [8]:
# 5. Install flash-attn (CRITICAL for model performance)
# ⚠️ This takes 10-15 minutes to compile CUDA kernels

print("🔧 Installing flash-attn...")
print("⏱️  This takes 10-15 minutes - please be patient\n")

# Install flash-attn (tail -5 shows last 5 lines for progress)
!pip install flash-attn --no-build-isolation 2>&1 | tail -5

print("\n" + "="*60)
print("⚠️  IMPORTANT: RESTART RUNTIME NOW!")
print("="*60)
print("\nGo to: Runtime > Restart runtime")
print("Then continue from Cell 6 (Clone Repositories)")
print("="*60)

🔧 Installing flash-attn...
⏱️  This takes 10-15 minutes - please be patient

  Created wheel for flash-attn: filename=flash_attn-2.8.3-cp312-cp312-linux_x86_64.whl size=255984554 sha256=51f6422861ed951b968428adc9fa7406027f73f2145be5e163810df6f459abea
  Stored in directory: /root/.cache/pip/wheels/3d/59/46/f282c12c73dd4bb3c2e3fe199f1a0d0f8cec06df0cccfeee27
Successfully built flash-attn
  Created wheel for flash-attn: filename=flash_attn-2.8.3-cp312-cp312-linux_x86_64.whl size=255984554 sha256=51f6422861ed951b968428adc9fa7406027f73f2145be5e163810df6f459abea
  Stored in directory: /root/.cache/pip/wheels/3d/59/46/f282c12c73dd4bb3c2e3fe199f1a0d0f8cec06df0cccfeee27
Successfully built flash-attn

⚠️  IMPORTANT: RESTART RUNTIME NOW!

Go to: Runtime > Restart runtime
Then continue from Cell 6 (Clone Repositories)

⚠️  IMPORTANT: RESTART RUNTIME NOW!

Go to: Runtime > Restart runtime
Then continue from Cell 6 (Clone Repositories)


In [ ]:
# 6. Clone Repositories and Install Requirements
# Run this cell AFTER restarting runtime (from Cell 5)
import os
from pathlib import Path

# Verify flash-attn is working after restart
import torch
print(f"✅ PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")
try:
    import flash_attn
    print(f"✅ flash-attn: {flash_attn.__version__}")
except ImportError:
    print("❌ flash-attn not found - go back and run Cells 4-5, then restart runtime")
    raise ImportError("flash-attn is required")

# Clone Evo-1 (check for .git folder or Evo_1 subfolder to verify actual clone)
if not Path("/content/Evo-1/.git").exists() and not Path("/content/Evo-1/Evo_1").exists():
    print("\n📦 Cloning Evo-1...")
    !git clone https://github.com/MINT-SJTU/Evo-1.git /content/Evo-1-temp
    !cp -r /content/Evo-1-temp/* /content/Evo-1/
    !cp -r /content/Evo-1-temp/.git /content/Evo-1/
    !rm -rf /content/Evo-1-temp
    print("   Installing Evo-1 requirements...")
    !pip install -q -r /content/Evo-1/Evo_1/requirements.txt
else:
    print("✅ Evo-1 already cloned")

# Clone LIBERO for the client (need the libero package)
if not Path("/content/LIBERO/.git").exists():
    print("\n📦 Cloning LIBERO (for client)...")
    !git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git /content/LIBERO
    print("   Installing LIBERO package...")
    !cd /content/LIBERO && pip install -q -e .
else:
    print("✅ LIBERO already cloned")

# Clone and install rrt-algorithms (REQUIRED for VLABench motion planning)
if not Path("/content/rrt-algorithms/.git").exists():
    print("\n📦 Cloning rrt-algorithms (VLABench dependency)...")
    !git clone https://github.com/motion-planning/rrt-algorithms.git /content/rrt-algorithms
    print("   Installing rrt-algorithms...")
    !cd /content/rrt-algorithms && pip install -q -e .
else:
    print("✅ rrt-algorithms already cloned")

# Clone VLABench (skip asset download - will use manually uploaded assets)
if not Path("/content/VLABench/.git").exists():
    print("\n📦 Cloning VLABench...")
    !git clone https://github.com/OpenMOSS/VLABench.git /content/VLABench
    print("   Installing VLABench...")
    !cd /content/VLABench && pip install -q -e .
    print("   ⚠️ Skipping asset download - use Cell 6b to set up manually uploaded assets")
else:
    print("✅ VLABench already cloned")
    # Ensure VLABench is installed even if already cloned
    !cd /content/VLABench && pip install -q -e .

# Create checkpoint directories (in case they don't exist)
os.makedirs("/content/Evo-1/checkpoints/libero", exist_ok=True)
os.makedirs("/content/Evo-1/checkpoints/metaworld", exist_ok=True)

print("\n✅ Repositories setup complete")
print("   ✅ rrt-algorithms installed (VLABench motion planning)")
print("\n➡️  If using VLABench, run Cell 6b to set up your uploaded assets")

✅ PyTorch: 2.5.1+cu121, CUDA: 12.1
✅ flash-attn: 2.8.3

📦 Cloning Evo-1...
Cloning into '/content/Evo-1-temp'...
remote: Enumerating objects: 714, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (30/30), done.
remote: Enumerating objects: 714, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 714 (delta 39), reused 25 (delta 19), pack-reused 665 (from 1)
Receiving objects: 100% (714/714), 6.47 MiB | 14.18 MiB/s, done.
Resolving deltas: 100% (190/190), done.
remote: Total 714 (delta 39), reused 25 (delta 19), pack-reused 665 (from 1)
Receiving objects: 100% (714/714), 6.47 MiB | 14.18 MiB/s, done.
Resolving deltas: 100% (190/190), done.
   Installing Evo-1 requirements...
   Installing Evo-1 requirements...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 32.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 32.6 MB/s eta 0:00:00a 0:00:01
  Pre

In [ ]:
# 6b. Setup VLABench Assets from Google Drive
import os
from pathlib import Path

print("📦 Setting up VLABench assets from Google Drive...")

# Create assets directory
assets_dir = Path("/content/VLABench/VLABench/assets")
assets_dir.mkdir(parents=True, exist_ok=True)

# Mount Google Drive
from google.colab import drive
if not Path("/content/drive").exists():
    print("📂 Mounting Google Drive...")
    drive.mount('/content/drive')

# Path to your uploaded files
DRIVE_PATH = "/content/drive/MyDrive/Research/URECA/VLABench"

obj_zip_path = f"{DRIVE_PATH}/obj.zip"
scenes_zip_path = f"{DRIVE_PATH}/scenes.zip"

# Extract obj.zip
if Path(obj_zip_path).exists():
    print(f"\n📥 Found obj.zip at {obj_zip_path}")
    if not (assets_dir / "obj").exists():
        print("   📂 Extracting obj.zip (this may take a few minutes)...")
        !unzip -q -o "{obj_zip_path}" -d {assets_dir}
        print("   ✅ obj.zip extracted")
    else:
        print("   ✅ obj/ already exists")
else:
    print(f"\n❌ obj.zip not found at {obj_zip_path}")
    print("   Please check the path and ensure obj.zip is uploaded")

# Extract scenes.zip
if Path(scenes_zip_path).exists():
    print(f"\n📥 Found scenes.zip at {scenes_zip_path}")
    if not (assets_dir / "scenes").exists():
        print("   📂 Extracting scenes.zip...")
        !unzip -q -o "{scenes_zip_path}" -d {assets_dir}
        print("   ✅ scenes.zip extracted")
    else:
        print("   ✅ scenes/ already exists")
else:
    print(f"\n❌ scenes.zip not found at {scenes_zip_path}")
    print("   Please check the path and ensure scenes.zip is uploaded")

# Verify extraction
print("\n" + "="*60)
print("📁 VLABench assets verification:")
print("="*60)

if (assets_dir / "obj").exists():
    obj_count = len(list((assets_dir / "obj").rglob("*")))
    print(f"   ✅ obj/ folder: {obj_count} files")
else:
    print("   ❌ obj/ folder missing")

if (assets_dir / "scenes").exists():
    scenes_count = len(list((assets_dir / "scenes").rglob("*")))
    print(f"   ✅ scenes/ folder: {scenes_count} files")
else:
    print("   ❌ scenes/ folder missing")

print("\n✅ VLABench assets setup complete")

In [ ]:
# 7. Download Checkpoints
print("📥 Downloading checkpoints...")

# We use huggingface_hub which is already in the base Colab environment
from huggingface_hub import snapshot_download
from pathlib import Path
import concurrent.futures

CHECKPOINTS = {
    "libero": {"repo_id": "MINT-SJTU/Evo1_LIBERO", "local_dir": "/content/Evo-1/checkpoints/libero"},
    "metaworld": {"repo_id": "MINT-SJTU/Evo1_MetaWorld", "local_dir": "/content/Evo-1/checkpoints/metaworld"}
}

def download_ckpt(name, config):
    if Path(f"{config['local_dir']}/config.json").exists():
        return f"✅ {name}: Already downloaded"
    snapshot_download(repo_id=config["repo_id"], local_dir=config["local_dir"], local_dir_use_symlinks=False)
    return f"✅ {name}: Downloaded"

with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(download_ckpt, name, config) for name, config in CHECKPOINTS.items()]
    for future in concurrent.futures.as_completed(futures):
        print(future.result())

print("\n✅ All checkpoints ready")
print("   Note: Using LIBERO checkpoint for VLABench zero-shot evaluation")

In [ ]:
# 8. Verify All Dependencies
import torch

print("📦 Verifying dependencies...\n")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not available'}")

try:
    import flash_attn
    print(f"flash-attn: {flash_attn.__version__} ✅")
except ImportError:
    print("flash-attn: ❌ Not found - go back to Cell 4, restart runtime, then continue")

try:
    import transformers
    print(f"transformers: {transformers.__version__} ✅")
except ImportError:
    print("transformers: ❌")

try:
    import mujoco
    print(f"mujoco: {mujoco.__version__} ✅")
except ImportError:
    print("mujoco: ❌")

print("\n✅ Dependency check complete")

In [ ]:
!ls Evo-1

In [ ]:
# 9. Create Multi-Server Script
print("📝 Creating multi-server script...")

# Read original server
with open("/content/Evo-1/Evo_1/scripts/Evo1_server.py", 'r') as f:
    original_server = f.read()

# Modify to accept command line args
modified_server = original_server
for old, new in [
    ('ckpt_dir = "Your/Path/To/Checkpoint"', 'ckpt_dir = args.checkpoint'),
    ('ckpt_dir = "/content/Evo-1/checkpoints/libero"', 'ckpt_dir = args.checkpoint'),
    ('port = 9000', 'port = args.port'),
    ('port = 9001', 'port = args.port'),
    ('port=9000', 'port=args.port'),
    ('port=9001', 'port=args.port'),
]:
    modified_server = modified_server.replace(old, new)

final_script = '''#!/usr/bin/env python3
"""Evo-1 Multi-Server Script - Supports multiple instances on different ports"""
import argparse
import sys
import os

parser = argparse.ArgumentParser(description="Evo-1 Server")
parser.add_argument("--port", type=int, required=True, help="Server port")
parser.add_argument("--checkpoint", type=str, required=True, help="Checkpoint directory")
parser.add_argument("--name", type=str, default="evo1", help="Server name for logging")
args = parser.parse_args()

print(f"[{args.name}] Starting server on port {args.port}")
print(f"[{args.name}] Checkpoint: {args.checkpoint}")

''' + modified_server

with open("/content/Evo-1/Evo_1/scripts/Evo1_multi_server.py", 'w') as f:
    f.write(final_script)

print("✅ Multi-server script created at /content/Evo-1/Evo_1/scripts/Evo1_multi_server.py")

In [ ]:
# 10. Create VLABench Client Script (LeRobot Format - Recommended)
# VLABench direct evaluation has mesh issues. Use LeRobot format instead.
print("📝 Creating VLABench evaluation scripts...")

# First, create a trajectory generation script (generates HDF5 files)
trajectory_gen_script = '''#!/usr/bin/env python3
"""
VLABench Trajectory Generation for Evo-1 Evaluation
Generates HDF5 trajectory data that can be converted to LeRobot format.

Usage:
    python vlabench_trajectory_gen.py --tasks select_toy select_fruit --n-sample 10
"""
import os
import sys
import argparse
import numpy as np
from tqdm import tqdm

os.environ["MUJOCO_GL"] = "egl"
sys.path.insert(0, "/content/VLABench")

def generate_trajectories(args):
    """Generate trajectory data for specified tasks."""
    # CRITICAL: Import robots and tasks FIRST to trigger registration decorators
    # This populates the registry before load_env tries to look up robot classes
    from VLABench.robots import *  # Registers franka, xarm, etc.
    from VLABench.tasks import *   # Registers all task classes
    
    from VLABench.envs import load_env
    from VLABench.utils.data_utils import save_single_data
    from VLABench.utils.skill_lib import SkillLib
    
    os.makedirs(args.save_dir, exist_ok=True)
    
    for task_name in args.tasks:
        print(f"\\n{'='*60}")
        print(f"Generating trajectories for: {task_name}")
        print(f"{'='*60}")
        
        task_save_dir = os.path.join(args.save_dir, task_name)
        os.makedirs(task_save_dir, exist_ok=True)
        
        success_count = 0
        for i in tqdm(range(args.n_sample), desc=f"[{task_name}]"):
            try:
                # Load environment
                env = load_env(task_name, random_init=True, run_mode="efficient")
                
                # Get expert skill sequence
                skill_sequence = env.task.get_expert_skill_sequence(env.physics)
                skill_lib = SkillLib(env)
                
                # Collect trajectory
                observations = []
                actions = []
                
                obs = env.reset()
                for skill in skill_sequence:
                    skill_name = skill["name"]
                    params = skill.get("params", {})
                    
                    # Execute skill and collect trajectory
                    traj = skill_lib.execute_skill(skill_name, **params)
                    for step in traj:
                        observations.append(step["observation"])
                        actions.append(step["action"])
                
                # Save trajectory
                data = {
                    "observation": observations,
                    "trajectory": np.array(actions),
                    "instruction": [env.task.get_instruction()],
                }
                
                filename = f"episode_{i:04d}.hdf5"
                save_single_data(data, task_save_dir, filename)
                success_count += 1
                
            except Exception as e:
                print(f"\\n   ⚠️ Episode {i} failed: {e}")
                continue
        
        print(f"   ✅ Generated {success_count}/{args.n_sample} episodes for {task_name}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--tasks", nargs="+", default=["select_toy", "select_fruit"])
    parser.add_argument("--n-sample", type=int, default=10)
    parser.add_argument("--save-dir", type=str, default="/content/vlabench_trajectories")
    args = parser.parse_args()
    
    generate_trajectories(args)
'''

# Create LeRobot conversion script (VLABench's convert_to_lerobot.py adapted)
lerobot_convert_script = '''#!/usr/bin/env python3
"""
Convert VLABench HDF5 trajectories to LeRobot v2.1 format.

LeRobot v2.1 format is supported by Evo-1 for training/evaluation.

Usage:
    python convert_vlabench_to_lerobot.py --dataset-name vlabench_eval --dataset-path /content/vlabench_trajectories
"""
import os
import sys
import argparse
import json
import numpy as np
import h5py
from scipy.spatial.transform import Rotation as R
from pathlib import Path

def quat2euler(quat, is_degree=False):
    """Convert quaternion to euler angles."""
    r = R.from_quat([quat[1], quat[2], quat[3], quat[0]])
    return r.as_euler('xyz', degrees=is_degree)

def get_all_hdf5_files(directory):
    """Get all HDF5 files in a directory recursively."""
    hdf5_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.hdf5'):
                hdf5_files.append(os.path.join(root, file))
    return sorted(hdf5_files)

def create_lerobot_dataset(args):
    """Convert HDF5 files to LeRobot dataset format."""
    from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
    
    print(f"Creating LeRobot dataset: {args.dataset_name}")
    print(f"Source path: {args.dataset_path}")
    
    # Create LeRobot dataset with VLABench features
    dataset = LeRobotDataset.create(
        repo_id=args.dataset_name,
        robot_type="franka",
        fps=10,
        features={
            "image": {
                "dtype": "image",
                "shape": (480, 480, 3),
                "names": ["height", "width", "channels"]
            },
            "wrist_image": {
                "dtype": "image",
                "shape": (480, 480, 3),
                "names": ["height", "width", "channels"]
            },
            "state": {
                "dtype": "float",
                "shape": (7,),
                "names": ["state"]
            },
            "actions": {
                "dtype": "float",
                "shape": (7,),
                "names": ["actions"]
            },
        },
        image_writer_processes=4,
        image_writer_threads=8
    )
    
    # Get task directories
    if args.task_list:
        tasks = args.task_list
    else:
        tasks = [d for d in os.listdir(args.dataset_path) 
                 if os.path.isdir(os.path.join(args.dataset_path, d))]
    
    print(f"Tasks to process: {tasks}")
    
    # Collect HDF5 files
    h5py_files = []
    for task in tasks:
        task_path = os.path.join(args.dataset_path, task)
        if os.path.exists(task_path):
            files = get_all_hdf5_files(task_path)[:args.max_files]
            h5py_files.extend(files)
    
    print(f"Total files to process: {len(h5py_files)}")
    
    # Process each HDF5 file
    for file_idx, file in enumerate(h5py_files):
        try:
            with h5py.File(file, "r") as f:
                for timestamp in f["data"].keys():
                    # Get robot frame position
                    try:
                        episode_config_bytes = np.asarray(f["data"][timestamp]["meta_info"]["episode_config"]).astype('S')
                        episode_config = json.loads(episode_config_bytes.item().decode('utf-8'))
                        robot_frame_pos = np.array(episode_config.get("robot", {}).get("position", [0, -0.4, 0.78]))
                    except:
                        robot_frame_pos = np.array([0, -0.4, 0.78])
                    
                    # Load data
                    images = f["data"][timestamp]["observation"]["rgb"][()]
                    ee_state = f["data"][timestamp]["observation"]["ee_state"][()]
                    actions = f["data"][timestamp]["trajectory"][()]
                    instruction = np.array(f["data"][timestamp]["instruction"])[0].decode("utf-8")
                    
                    # Process ee_state
                    ee_pos, ee_quat, gripper = ee_state[:, :3], ee_state[:, 3:7], ee_state[:, 7]
                    ee_euler = np.array([quat2euler(q) for q in ee_quat])
                    ee_pos -= robot_frame_pos
                    ee_state_processed = np.concatenate([ee_pos, ee_euler, gripper.reshape(-1, 1)], axis=1)
                    
                    # Add frames
                    for i in range(images.shape[0]):
                        action = actions[i]
                        # Binarize gripper
                        if action[-1] > 0.03:
                            action = np.concatenate([action[:6], np.array([1])])
                        else:
                            action = np.concatenate([action[:6], np.array([0])])
                        
                        dataset.add_frame({
                            "image": images[i][2],  # front camera
                            "wrist_image": images[i][3],  # wrist camera
                            "state": ee_state_processed[i],
                            "actions": action
                        })
                    
                    dataset.save_episode(task=instruction)
                    
        except Exception as e:
            print(f"\\n   ⚠️ Error processing {file}: {e}")
            continue
        
        if (file_idx + 1) % 10 == 0:
            print(f"   Processed {file_idx + 1}/{len(h5py_files)} files")
    
    print("\\n📊 Computing statistics...")
    dataset.consolidate(run_compute_stats=True)
    print(f"\\n✅ LeRobot dataset created at: {os.environ.get('HF_LEROBOT_HOME', '~/.cache/huggingface/lerobot')}/{args.dataset_name}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset-name", type=str, default="vlabench_eval")
    parser.add_argument("--dataset-path", type=str, default="/content/vlabench_trajectories")
    parser.add_argument("--max-files", type=int, default=500)
    parser.add_argument("--task-list", nargs="+", default=None)
    args = parser.parse_args()
    
    create_lerobot_dataset(args)
'''

# Create Evo-1 LeRobot evaluation config
evo1_lerobot_config = '''# Evo-1 LeRobot Dataset Config for VLABench
# Place this at: /content/vlabench_lerobot_config.yaml

max_action_dim: 7
max_state_dim: 7
max_views: 3

data_groups:
  franka:
    vlabench:
      path: ${HF_LEROBOT_HOME}/vlabench_eval
      views:
        - image
        - wrist_image
      action_dim: 7
      state_dim: 7
'''

# Create simple WebSocket client for direct evaluation (fallback)
vlabench_ws_client = '''#!/usr/bin/env python3
"""
VLABench WebSocket Client for Evo-1 - Simple evaluation without Evaluator class.

This avoids mesh compilation issues by using a simpler evaluation loop.
Falls back to specific tasks that are known to work.

Usage:
    python vlabench_simple_client.py --host localhost --port 9003 --tasks select_toy
"""
import argparse
import json
import numpy as np
import websocket
import math
import os
import sys
import traceback

os.environ["MUJOCO_GL"] = "egl"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

sys.path.insert(0, "/content/VLABench")

def encode_image_array(img_array):
    return img_array.astype(np.uint8).tolist()

def quat2axisangle(quat):
    if quat[3] > 1.0: quat[3] = 1.0
    elif quat[3] < -1.0: quat[3] = -1.0
    den = np.sqrt(1.0 - quat[3] * quat[3])
    if math.isclose(den, 0.0): return np.zeros(3)
    return (quat[:3] * 2.0 * math.acos(quat[3])) / den

class Evo1Client:
    """WebSocket client for Evo-1 VLA model."""
    
    def __init__(self, host, port):
        self.url = f"ws://{host}:{port}"
        self.ws = websocket.create_connection(self.url)
        print(f"✅ Connected to Evo-1 at {self.url}")

    def predict(self, obs, instruction):
        """Get action prediction from Evo-1."""
        import cv2
        
        # Get images from observation
        img = obs.get("rgb", np.zeros((448, 448, 3), dtype=np.uint8))
        if img.ndim == 4:  # Multiple cameras
            img = img[2] if img.shape[0] > 2 else img[0]  # Use front camera
        
        wrist_img = obs.get("wrist", np.zeros((448, 448, 3), dtype=np.uint8))
        if wrist_img.ndim == 4:
            wrist_img = wrist_img[3] if wrist_img.shape[0] > 3 else wrist_img[-1]
        
        # Resize to 448x448
        if img.shape[:2] != (448, 448):
            img = cv2.resize(img, (448, 448))
        if wrist_img.shape[:2] != (448, 448):
            wrist_img = cv2.resize(wrist_img, (448, 448))
        
        dummy = np.zeros((448, 448, 3), dtype=np.uint8)
        
        # Get state
        ee_state = obs.get("ee_state", np.zeros(8))
        if len(ee_state) >= 7:
            pos = ee_state[:3]
            quat = ee_state[3:7]
            gripper = ee_state[7:] if len(ee_state) > 7 else [0]
            axis_angle = quat2axisangle(quat)
            state = np.concatenate((pos, axis_angle, gripper)).tolist()
        else:
            state = [0]*7

        data = {
            "image": [encode_image_array(img), encode_image_array(wrist_img), encode_image_array(dummy)],
            "state": state,
            "prompt": instruction,
            "image_mask": [1, 1, 0],
            "action_mask": [1] * 7 + [0] * 17,
        }
        
        self.ws.send(json.dumps(data))
        result = self.ws.recv()
        actions = json.loads(result)
        return np.array(actions[0])
    
    def close(self):
        self.ws.close()

# List of tasks known to have working assets (avoid mesh issues)
WORKING_TASKS = [
    # Primitive tasks with simple objects
    "select_toy",
    "select_fruit", 
    "select_drink",
    "select_ingredient",
    "add_condiment",
    # Add more as tested
]

def evaluate_task(client, task_name, n_episodes=5, max_steps=200):
    """Evaluate a single task."""
    from VLABench.envs import load_env
    
    results = {"task": task_name, "episodes": [], "success_rate": 0.0}
    successes = 0
    
    for ep in range(n_episodes):
        try:
            # Load environment with error handling
            env = load_env(task_name, random_init=True, run_mode="eval", eval=False)
            instruction = env.task.get_instruction()
            
            print(f"   Episode {ep+1}/{n_episodes}: {instruction[:50]}...")
            
            obs = env.reset().observation
            done = False
            step = 0
            
            while not done and step < max_steps:
                # Get observation dict
                obs_dict = {
                    "rgb": obs.get("rgb", obs.get("image", np.zeros((4, 480, 480, 3)))),
                    "ee_state": obs.get("ee_state", np.zeros(8)),
                    "wrist": obs.get("wrist", obs.get("rgb", np.zeros((4, 480, 480, 3))))
                }
                
                # Get action from Evo-1
                action = client.predict(obs_dict, instruction)
                
                # Step environment
                timestep = env.step(action)
                obs = timestep.observation
                done = timestep.last()
                step += 1
            
            # Check success
            success = env.task.check_success(env.physics)
            if success:
                successes += 1
                print(f"      ✅ Success! ({step} steps)")
            else:
                print(f"      ❌ Failed ({step} steps)")
            
            results["episodes"].append({"success": success, "steps": step})
            
        except Exception as e:
            print(f"      ⚠️ Error: {e}")
            results["episodes"].append({"success": False, "error": str(e)})
    
    results["success_rate"] = successes / n_episodes if n_episodes > 0 else 0
    return results

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--host", default="localhost")
    parser.add_argument("--port", type=int, default=9003)
    parser.add_argument("--tasks", nargs="+", default=None,
                        help="Tasks to evaluate (default: use WORKING_TASKS)")
    parser.add_argument("--n-episodes", type=int, default=5)
    parser.add_argument("--max-steps", type=int, default=200)
    parser.add_argument("--save-dir", type=str, default="/content/logs/vlabench")
    args = parser.parse_args()
    
    print("="*60)
    print("VLABench Simple Client for Evo-1")
    print("="*60)
    
    # Import VLABench
    try:
        import VLABench
        print(f"✅ VLABench imported from {VLABench.__file__}")
    except ImportError as e:
        print(f"❌ VLABench import error: {e}")
        sys.exit(1)
    
    # Import tasks to trigger registration
    try:
        from VLABench.tasks import *
        from VLABench.robots import *
        from VLABench.utils.register import register
        print(f"✅ Loaded {len(register._tasks)} tasks from registry")
    except Exception as e:
        print(f"⚠️ Task import warning: {e}")
    
    # Determine tasks to evaluate
    if args.tasks:
        tasks = args.tasks
    else:
        tasks = WORKING_TASKS
    
    print(f"\\nTasks to evaluate: {tasks}")
    print(f"Episodes per task: {args.n_episodes}")
    
    os.makedirs(args.save_dir, exist_ok=True)
    
    # Connect to Evo-1
    print(f"\\n🔌 Connecting to Evo-1 at {args.host}:{args.port}...")
    try:
        client = Evo1Client(args.host, args.port)
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        sys.exit(1)
    
    # Evaluate each task
    all_results = {}
    for task in tasks:
        print(f"\\n{'='*60}")
        print(f"Evaluating: {task}")
        print(f"{'='*60}")
        
        try:
            result = evaluate_task(client, task, args.n_episodes, args.max_steps)
            all_results[task] = result
            print(f"\\n   Success Rate: {result['success_rate']*100:.1f}%")
        except Exception as e:
            print(f"\\n   ❌ Task failed: {e}")
            traceback.print_exc()
            all_results[task] = {"error": str(e), "success_rate": 0}
    
    # Print summary
    print(f"\\n{'='*60}")
    print("VLABench Results Summary")
    print(f"{'='*60}")
    
    total_success = 0
    total_episodes = 0
    for task, result in all_results.items():
        sr = result.get("success_rate", 0) * 100
        print(f"   {task}: {sr:.1f}%")
        if "success_rate" in result:
            total_success += result["success_rate"] * args.n_episodes
            total_episodes += args.n_episodes
    
    if total_episodes > 0:
        avg_sr = total_success / total_episodes * 100
        print(f"\\n   Average: {avg_sr:.1f}%")
    
    # Save results
    result_file = os.path.join(args.save_dir, "results.json")
    with open(result_file, "w") as f:
        json.dump(all_results, f, indent=2, default=str)
    print(f"\\nResults saved to {result_file}")
    
    client.close()

if __name__ == "__main__":
    main()
'''

# Save all scripts
with open("/content/vlabench_trajectory_gen.py", 'w') as f:
    f.write(trajectory_gen_script)

with open("/content/convert_vlabench_to_lerobot.py", 'w') as f:
    f.write(lerobot_convert_script)

with open("/content/vlabench_lerobot_config.yaml", 'w') as f:
    f.write(evo1_lerobot_config)

with open("/content/vlabench_client.py", 'w') as f:
    f.write(vlabench_ws_client)

print("✅ Created VLABench evaluation scripts:")
print("")
print("📋 Option 1: LeRobot Format (Recommended for Evo-1)")
print("   1. vlabench_trajectory_gen.py - Generate HDF5 trajectories")
print("   2. convert_vlabench_to_lerobot.py - Convert to LeRobot v2.1")
print("   3. vlabench_lerobot_config.yaml - Evo-1 dataset config")
print("")
print("📋 Option 2: Direct WebSocket Client (Simple, may have mesh issues)")
print("   vlabench_client.py - Direct evaluation with known working tasks")
print("")
print("⚠️ Note: VLABench has mesh volume issues with some assets.")
print("   Use --tasks to specify tasks with working assets.")
print("   Known working: select_toy, select_fruit, select_drink")

In [ ]:
# 10b. Run VLABench Evaluation
# Choose your evaluation approach:
# - Option A: LeRobot format (data-driven, avoids MuJoCo mesh issues)
# - Option B: Direct client (interactive, but may hit mesh issues)

# ============================================================
# OPTION A: LeRobot Format Evaluation (Recommended)
# ============================================================
# This approach:
# 1. Generates trajectory data from expert demonstrations
# 2. Converts to LeRobot v2.1 format (Evo-1's native format)
# 3. Compares Evo-1 predictions against ground truth actions
#
# Benefits: Avoids MuJoCo mesh issues, reproducible, uses official Evo-1 format

USE_LEROBOT_FORMAT = True  # Set to False to use direct evaluation

if USE_LEROBOT_FORMAT:
    print("📊 Using LeRobot Format Evaluation")
    print("=" * 60)
    
    # Step 1: Get available tasks from VLABench assets
    import os
    import sys
    sys.path.insert(0, "/content/VLABench")
    
    # Find tasks with existing trajectory data in Google Drive assets
    asset_path = "/content/vlabench_assets"
    data_path = os.path.join(asset_path, "data")
    
    available_tasks = []
    if os.path.exists(data_path):
        for item in os.listdir(data_path):
            task_dir = os.path.join(data_path, item)
            if os.path.isdir(task_dir):
                # Check if it has HDF5 files
                hdf5_files = [f for f in os.listdir(task_dir) if f.endswith('.hdf5')]
                if hdf5_files:
                    available_tasks.append(item)
                    print(f"   Found: {item} ({len(hdf5_files)} files)")
    
    if available_tasks:
        print(f"\n✅ Found {len(available_tasks)} tasks with existing trajectory data")
        print("   Converting to LeRobot format...")
        
        # Run conversion
        !cd /content && python convert_vlabench_to_lerobot.py \
            --dataset-name vlabench_evo1_eval \
            --dataset-path "{data_path}" \
            --task-list {' '.join(available_tasks[:10])} \
            --max-files 50
        
        print("\n✅ LeRobot dataset ready for Evo-1 evaluation")
        
    else:
        print("⚠️ No pre-existing trajectory data found.")
        print("   Generating trajectories for a few tasks...")
        
        # Generate trajectories for tasks known to work
        EVAL_TASKS = ["select_toy", "select_fruit", "select_drink"]
        
        # Simple trajectory collection (without full SkillLib for now)
        import numpy as np
        import h5py
        from tqdm import tqdm
        
        try:
            from VLABench.envs import load_env
            
            traj_save_dir = "/content/vlabench_trajectories"
            os.makedirs(traj_save_dir, exist_ok=True)
            
            for task_name in EVAL_TASKS:
                try:
                    task_dir = os.path.join(traj_save_dir, task_name)
                    os.makedirs(task_dir, exist_ok=True)
                    
                    print(f"\nGenerating for {task_name}...")
                    env = load_env(task_name, random_init=False, run_mode="efficient")
                    
                    # Collect 3 episodes with random actions (for demo)
                    for ep in range(3):
                        try:
                            obs = env.reset().observation
                            
                            observations = []
                            actions = []
                            
                            # Collect 50 random steps
                            for step in range(50):
                                action = np.random.randn(7) * 0.01
                                action[-1] = 0  # gripper open
                                
                                observations.append({
                                    "rgb": obs.get("rgb", np.zeros((4, 480, 480, 3))),
                                    "ee_state": obs.get("ee_state", np.zeros(8))
                                })
                                actions.append(action)
                                
                                timestep = env.step(action)
                                obs = timestep.observation
                            
                            # Save as HDF5
                            filename = os.path.join(task_dir, f"episode_{ep:04d}.hdf5")
                            with h5py.File(filename, 'w') as f:
                                grp = f.create_group("data")
                                ep_grp = grp.create_group("0")
                                obs_grp = ep_grp.create_group("observation")
                                
                                # Stack observations
                                rgb_stack = np.array([o["rgb"] for o in observations])
                                state_stack = np.array([o["ee_state"] for o in observations])
                                
                                obs_grp.create_dataset("rgb", data=rgb_stack)
                                obs_grp.create_dataset("ee_state", data=state_stack)
                                ep_grp.create_dataset("trajectory", data=np.array(actions))
                                
                                instruction = env.task.get_instruction()
                                ep_grp.create_dataset("instruction", data=np.array([instruction.encode()]))
                            
                            print(f"   ✅ Saved episode {ep}")
                            
                        except Exception as e:
                            print(f"   ⚠️ Episode {ep} error: {e}")
                            
                except Exception as e:
                    print(f"   ❌ Task {task_name} failed: {e}")
                    
            print("\n✅ Trajectory data generated")
            print("   Run convert_vlabench_to_lerobot.py to create LeRobot dataset")
            
        except Exception as e:
            print(f"❌ Trajectory generation failed: {e}")
            import traceback
            traceback.print_exc()

# ============================================================
# OPTION B: Direct WebSocket Evaluation
# ============================================================
# This uses the simple client to evaluate directly against VLABench env
# May encounter MuJoCo mesh errors for some tasks

else:
    print("🔌 Using Direct WebSocket Evaluation")
    print("=" * 60)
    
    # Get list of available tasks from registry
    from VLABench.utils.register import register
    
    # Filter to tasks that are known to work (avoid mesh issues)
    SAFE_TASKS = [
        "select_toy",
        "select_fruit",
        "select_drink",
        "select_ingredient",
    ]
    
    available = [t for t in SAFE_TASKS if t in register._tasks]
    print(f"Available safe tasks: {available}")
    
    # Run evaluation
    task_arg = " ".join(available) if available else "select_toy"
    
    !cd /content && python vlabench_client.py \
        --host {EVO1_HOST} \
        --port {EVO1_PORT_VLABENCH} \
        --tasks {task_arg} \
        --n-episodes 3 \
        --max-steps 100 \
        --save-dir /content/logs/vlabench
    
    print("\n✅ VLABench evaluation complete!")

In [ ]:
# 11. Patch Client Scripts to Use Localhost
# Using Python to patch (sed can corrupt the file)
print("📝 Patching client scripts...")

import re

# Patch LIBERO client - fix the SERVER_URL inside the Args class
libero_client = "/content/Evo-1/LIBERO_evaluation/libero_client_4tasks.py"
with open(libero_client, 'r') as f:
    content = f.read()

# Replace SERVER_URL assignment (handles various formats)
content = re.sub(
    r'SERVER_URL\s*=\s*["\'][^"\']*["\']',
    'SERVER_URL = "ws://localhost:9001"',
    content
)
# Also set MUJOCO_GL for headless rendering
content = content.replace('os.environ["MUJOCO_GL"] = "osmesa"', 'os.environ["MUJOCO_GL"] = "egl"')
content = content.replace('os.environ["MUJOCO_GL"] = "osmes"', 'os.environ["MUJOCO_GL"] = "egl"')

with open(libero_client, 'w') as f:
    f.write(content)
print("   ✅ LIBERO client patched to use ws://localhost:9001")

# Patch MetaWorld client
mw_client = "/content/Evo-1/MetaWorld_evaluation/mt50_evo1_client_prompt.py"
with open(mw_client, 'r') as f:
    content = f.read()

content = re.sub(
    r'SERVER_URL\s*=\s*["\'][^"\']*["\']',
    'SERVER_URL = "ws://localhost:9002"',
    content
)
# Disable display for headless
content = re.sub(r'SHOW_WINDOW\s*=\s*True', 'SHOW_WINDOW = False', content)
content = re.sub(r'SAVE_VIDEO\s*=\s*True', 'SAVE_VIDEO = False', content)

with open(mw_client, 'w') as f:
    f.write(content)
print("   ✅ MetaWorld client patched to use ws://localhost:9002 (headless mode)")

print("\n✅ All client scripts patched")

In [ ]:
# 12. Run All Benchmarks in Parallel
import subprocess
import time
import os

# Set environment for headless rendering
os.environ["MUJOCO_GL"] = "egl"
os.environ["DISPLAY"] = ""
os.environ["ROBOSUITE_MACROS"] = "~/.robosuite/macros_private.py"

# Pre-create robosuite macros to avoid interactive prompt
robosuite_macros_dir = os.path.expanduser("~/.robosuite")
os.makedirs(robosuite_macros_dir, exist_ok=True)
macros_file = os.path.join(robosuite_macros_dir, "macros_private.py")
if not os.path.exists(macros_file):
    with open(macros_file, 'w') as f:
        f.write('# Robosuite macros - auto-generated\n')
        f.write('import robosuite.macros as macros\n')
        f.write('macros.SIMULATION_TIMESTEP = 0.002\n')
        f.write('macros.SHOW_WARNINGS = False\n')
    print("✅ Created robosuite macros file")

# Create logs directory
os.makedirs("/content/logs", exist_ok=True)
os.makedirs("/content/logs/vlabench", exist_ok=True)

# Configuration - run servers on different ports
SERVERS = [
    {"name": "libero", "port": 9001, "ckpt": "/content/Evo-1/checkpoints/libero"},
    {"name": "metaworld", "port": 9002, "ckpt": "/content/Evo-1/checkpoints/metaworld"},
    {"name": "vlabench", "port": 9003, "ckpt": "/content/Evo-1/checkpoints/libero"}  # Zero-shot
]

processes = []

print("="*60)
print("STARTING EVO-1 SERVERS")
print("="*60)

for s in SERVERS:
    cmd = f"python /content/Evo-1/Evo_1/scripts/Evo1_multi_server.py --port {s['port']} --checkpoint {s['ckpt']} --name {s['name']}"
    log = open(f"/content/logs/server_{s['name']}.log", "w")
    p = subprocess.Popen(cmd, shell=True, stdout=log, stderr=log)
    processes.append((s['name'], p))
    print(f"   🚀 Started {s['name']} server on port {s['port']}")

print("\n⏳ Waiting 90 seconds for models to load...")
time.sleep(90)

print("\n" + "="*60)
print("STARTING BENCHMARK CLIENTS")
print("="*60)

# Environment for all clients
client_env = {
    **os.environ,
    "MUJOCO_GL": "egl",
    "DISPLAY": "",
    "PYTHONPATH": "/content/LIBERO:/content/VLABench",
    "ROBOSUITE_MACROS": os.path.expanduser("~/.robosuite/macros_private.py"),
}

# LIBERO Client
cmd_libero = "cd /content/LIBERO && python /content/Evo-1/LIBERO_evaluation/libero_client_4tasks.py"
log_libero = open("/content/logs/client_libero.log", "w")
p_libero = subprocess.Popen(cmd_libero, shell=True, stdout=log_libero, stderr=log_libero, 
                            stdin=subprocess.DEVNULL,
                            env=client_env)
processes.append(("libero_client", p_libero))
print("   🔬 Started LIBERO client")

# MetaWorld Client
cmd_mw = "cd /content/Evo-1/MetaWorld_evaluation && python mt50_evo1_client_prompt.py"
log_mw = open("/content/logs/client_metaworld.log", "w")
p_mw = subprocess.Popen(cmd_mw, shell=True, stdout=log_mw, stderr=log_mw,
                        stdin=subprocess.DEVNULL,
                        env=client_env)
processes.append(("metaworld_client", p_mw))
print("   🔬 Started MetaWorld client")

# VLABench Client - use simple client with safe tasks to avoid mesh issues
# Only evaluates tasks with known working assets
VLA_SAFE_TASKS = "select_toy select_fruit select_drink"
cmd_vla = f"cd /content && python vlabench_client.py --host localhost --port 9003 --tasks {VLA_SAFE_TASKS} --n-episodes 3 --max-steps 100 --save-dir /content/logs/vlabench"
log_vla = open("/content/logs/client_vlabench.log", "w")
p_vla = subprocess.Popen(cmd_vla, shell=True, stdout=log_vla, stderr=log_vla,
                         stdin=subprocess.DEVNULL,
                         env=client_env)
processes.append(("vlabench_client", p_vla))
print("   🔬 Started VLABench client (safe tasks only)")

print("\n" + "="*60)
print("✅ All benchmarks running in parallel!")
print("="*60)
print("\n📄 Log files in /content/logs/")
print("   - server_libero.log, server_metaworld.log, server_vlabench.log")
print("   - client_libero.log, client_metaworld.log, client_vlabench.log")
print("\n⚠️ VLABench Note: Only testing tasks with verified assets.")
print("   For full VLABench, use LeRobot format (Cell 10b with USE_LEROBOT_FORMAT=True)")
print("\n⏳ Monitoring... (Stop this cell to kill all processes)")

try:
    while True:
        time.sleep(30)
        print("\n--- Status Update ---")
        for name, p in processes:
            status = "Running" if p.poll() is None else f"Exited ({p.returncode})"
            print(f"   {name}: {status}")
        print("\n--- Last log lines ---")
        !tail -n 1 /content/logs/client_*.log 2>/dev/null || echo "No client logs yet"
except KeyboardInterrupt:
    print("\n🛑 Stopping all processes...")
    for name, p in processes:
        p.terminate()
    print("✅ All processes stopped")

In [ ]:
!pip install bddl

In [ ]:
!ls logs/

In [ ]:
!cat logs/client_vlabench.log

In [ ]:
# 13. View Results (Run after benchmarks complete)
print("📊 Benchmark Results")
print("="*60)

print("\n--- LIBERO Results ---")
!tail -n 20 /content/logs/client_libero.log 2>/dev/null || echo "No LIBERO results yet"

print("\n--- MetaWorld Results ---")
!tail -n 20 /content/logs/client_metaworld.log 2>/dev/null || echo "No MetaWorld results yet"

print("\n--- VLABench Results ---")
!tail -n 20 /content/logs/client_vlabench.log 2>/dev/null || echo "No VLABench results yet"

print("\n" + "="*60)
print("Server Logs (check for errors):")
print("="*60)
!tail -n 5 /content/logs/server_*.log 2>/dev/null || echo "No server logs"

In [ ]:
# 14. Download All Logs
# Creates a zip file with all logs named log_<timestamp>_all.zip and downloads it

import os
import zipfile
from datetime import datetime

# Create timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"log_{timestamp}_all.zip"
zip_path = f"/content/{zip_filename}"

print(f"📦 Creating {zip_filename}...")

# Create zip file with all logs
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    logs_dir = "/content/logs"
    if os.path.exists(logs_dir):
        for root, dirs, files_list in os.walk(logs_dir):
            for filename in files_list:
                filepath = os.path.join(root, filename)
                arcname = os.path.relpath(filepath, logs_dir)
                zipf.write(filepath, arcname)
                print(f"   ✅ Added: {arcname}")
    
    # Also include VLABench results if they exist
    vlabench_results = "/content/logs/vlabench/results.json"
    if os.path.exists(vlabench_results):
        zipf.write(vlabench_results, "vlabench_results.json")
        print(f"   ✅ Added: vlabench_results.json")
    
    # Include LeRobot dataset info if it exists
    lerobot_home = os.environ.get("HF_LEROBOT_HOME", os.path.expanduser("~/.cache/huggingface/lerobot"))
    vlabench_lerobot = os.path.join(lerobot_home, "vlabench_evo1_eval")
    if os.path.exists(vlabench_lerobot):
        # Include meta files (not the full dataset)
        meta_dir = os.path.join(vlabench_lerobot, "meta")
        if os.path.exists(meta_dir):
            for f in os.listdir(meta_dir):
                zipf.write(os.path.join(meta_dir, f), f"lerobot_meta/{f}")
                print(f"   ✅ Added: lerobot_meta/{f}")
    
    # Include generated trajectory info
    traj_dir = "/content/vlabench_trajectories"
    if os.path.exists(traj_dir):
        for task in os.listdir(traj_dir):
            task_dir = os.path.join(traj_dir, task)
            if os.path.isdir(task_dir):
                count = len([f for f in os.listdir(task_dir) if f.endswith('.hdf5')])
                if count > 0:
                    # Just add a summary, not the full data
                    summary = f"{task}: {count} episodes\n"
                    zipf.writestr(f"trajectory_summary/{task}.txt", summary)
                    print(f"   ✅ Added: trajectory_summary/{task}.txt")

print(f"\n✅ Created {zip_filename}")

# Download the file
from google.colab import files
print(f"\n📥 Downloading {zip_filename}...")
files.download(zip_path)

print("\n✅ Download started! Check your browser downloads.")